# Open-Meteo all variables experiment (hourly + daily)

This notebook tests and downloads **as many Open-Meteo variables as possible** for a location, then exports CSV files to compare new covariates for your 24h price model.

Flow:
1. Define a broad candidate list from Open-Meteo docs.
2. Validate which variables are accepted for the selected location/model.
3. Download final hourly and daily datasets.
4. Save CSV outputs for experimentation.

In [1]:
%pip install -q requests pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import time
import requests
import pandas as pd

In [3]:
# --- Project / API config ---
PROJECT_ROOT = Path.cwd().parent.parent.parent.parent
OUTPUT_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Use archive endpoint for historical ranges (e.g. full year 2025)
URL = "https://archive-api.open-meteo.com/v1/archive"
TIMEZONE = "GMT"

# Date interval can be adjusted
START_DATE = "2025-01-01"
END_DATE = "2025-12-31"

# Add as many points as needed
LOCATIONS = [
    {"location_id": "uk_london", "latitude": 51.5074, "longitude": -0.1278},
    {"location_id": "uk_birmingham", "latitude": 52.4862, "longitude": -1.8904},
    {"location_id": "uk_manchester", "latitude": 53.4808, "longitude": -2.2426},
]

COMMON_PARAMS = {
    "timezone": TIMEZONE,
    "start_date": START_DATE,
    "end_date": END_DATE,
}

print("Base params:", COMMON_PARAMS)
print(f"Locations configured: {len(LOCATIONS)}")
display(pd.DataFrame(LOCATIONS))

Base params: {'timezone': 'GMT', 'start_date': '2025-01-01', 'end_date': '2025-12-31'}
Locations configured: 3


,location_id,latitude,longitude
0,uk_london,51.5074,-0.1278
1,uk_birmingham,52.4862,-1.8904
2,uk_manchester,53.4808,-2.2426


In [4]:
# --- Candidate variable lists from Open-Meteo docs (broad set) ---
HOURLY_CANDIDATES = [
    "temperature_2m", "relative_humidity_2m", "dew_point_2m", "apparent_temperature",
    "precipitation_probability", "precipitation", "rain", "showers", "snowfall", "snow_depth",
    "weather_code", "pressure_msl", "surface_pressure",
    "cloud_cover", "cloud_cover_low", "cloud_cover_mid", "cloud_cover_high",
    "visibility", "evapotranspiration", "et0_fao_evapotranspiration", "vapour_pressure_deficit",
    "wind_speed_10m", "wind_speed_80m", "wind_speed_120m", "wind_speed_180m",
    "wind_direction_10m", "wind_direction_80m", "wind_direction_120m", "wind_direction_180m",
    "wind_gusts_10m",
    "shortwave_radiation", "direct_radiation", "direct_normal_irradiance", "diffuse_radiation",
    "global_tilted_irradiance", "sunshine_duration",
    "is_day", "freezing_level_height", "cape",
    "soil_temperature_0cm", "soil_temperature_6cm", "soil_temperature_18cm", "soil_temperature_54cm",
    "soil_moisture_0_to_1cm", "soil_moisture_1_to_3cm", "soil_moisture_3_to_9cm",
    "soil_moisture_9_to_27cm", "soil_moisture_27_to_81cm",
]

DAILY_CANDIDATES = [
    "weather_code",
    "temperature_2m_max", "temperature_2m_mean", "temperature_2m_min",
    "apparent_temperature_max", "apparent_temperature_mean", "apparent_temperature_min",
    "sunrise", "sunset", "daylight_duration", "sunshine_duration",
    "precipitation_sum", "rain_sum", "showers_sum", "snowfall_sum",
    "precipitation_hours", "precipitation_probability_max", "precipitation_probability_mean", "precipitation_probability_min",
    "wind_speed_10m_max", "wind_gusts_10m_max", "wind_direction_10m_dominant",
    "shortwave_radiation_sum", "et0_fao_evapotranspiration",
    "uv_index_max", "uv_index_clear_sky_max",
]

print(f"Hourly candidates: {len(HOURLY_CANDIDATES)}")
print(f"Daily candidates: {len(DAILY_CANDIDATES)}")

Hourly candidates: 48
Daily candidates: 26


In [5]:
def call_forecast(params: dict, timeout: int = 60) -> dict:
    response = requests.get(URL, params=params, timeout=timeout)
    if response.status_code >= 400:
        reason = ""
        try:
            payload = response.json()
            reason = payload.get("reason", "") if isinstance(payload, dict) else ""
        except Exception:  # noqa: BLE001
            reason = ""
        detail = f" | reason: {reason}" if reason else ""
        raise ValueError(f"HTTP {response.status_code} for {response.url}{detail}")

    payload = response.json()
    if payload.get("error"):
        raise ValueError(payload.get("reason", "Unknown Open-Meteo error"))
    return payload


def location_params(location: dict, **extra: object) -> dict:
    """Build request params for a single location plus optional extra query args."""
    return {
        **COMMON_PARAMS,
        "latitude": location["latitude"],
        "longitude": location["longitude"],
        **extra,
    }


def validate_variables_for_location(
    location: dict,
    kind: str,
    candidates: list[str],
    sleep_s: float = 0.02,
    timeout: int = 30,
) -> tuple[list[str], dict[str, str]]:
    """Validate variables one by one for a specific location."""
    valid: list[str] = []
    rejected: dict[str, str] = {}

    for variable in candidates:
        test_params = location_params(location, **{kind: variable})
        try:
            _ = call_forecast(test_params, timeout=timeout)
            valid.append(variable)
        except Exception as exc:  # noqa: BLE001
            rejected[variable] = str(exc)
        time.sleep(sleep_s)

    return valid, rejected

In [6]:
# Validate available variables for each location, then keep intersection for a consistent schema
validation_by_location: dict[str, dict[str, list[str] | dict[str, str]]] = {}

for location in LOCATIONS:
    loc_id = location["location_id"]
    print(f"Validating variables for {loc_id} ({location['latitude']}, {location['longitude']}) ...")

    loc_valid_hourly, loc_rejected_hourly = validate_variables_for_location(
        location, "hourly", HOURLY_CANDIDATES
    )
    loc_valid_daily, loc_rejected_daily = validate_variables_for_location(
        location, "daily", DAILY_CANDIDATES
    )

    validation_by_location[loc_id] = {
        "valid_hourly": loc_valid_hourly,
        "valid_daily": loc_valid_daily,
        "rejected_hourly": loc_rejected_hourly,
        "rejected_daily": loc_rejected_daily,
    }
    print(f"  hourly valid: {len(loc_valid_hourly)} | daily valid: {len(loc_valid_daily)}")

valid_hourly_lists = [set(v["valid_hourly"]) for v in validation_by_location.values()]
valid_daily_lists = [set(v["valid_daily"]) for v in validation_by_location.values()]

valid_hourly = sorted(set.intersection(*valid_hourly_lists)) if valid_hourly_lists else []
valid_daily = sorted(set.intersection(*valid_daily_lists)) if valid_daily_lists else []

print(f"\nCommon valid hourly variables across all points: {len(valid_hourly)}")
print(valid_hourly)
print(f"\nCommon valid daily variables across all points: {len(valid_daily)}")
print(valid_daily)

if not valid_hourly and not valid_daily:
    sample_errors = {}
    for loc_id, details in validation_by_location.items():
        merged_rejected = {**details["rejected_hourly"], **details["rejected_daily"]}
        if merged_rejected:
            sample_errors[loc_id] = dict(list(merged_rejected.items())[:5])
    raise ValueError(
        "No valid variables found for any location. Check URL/date range and API availability. "
        f"Sample errors: {sample_errors}"
    )

Validating variables for uk_london (51.5074, -0.1278) ...
  hourly valid: 42 | daily valid: 21
Validating variables for uk_birmingham (52.4862, -1.8904) ...
  hourly valid: 38 | daily valid: 21
Validating variables for uk_manchester (53.4808, -2.2426) ...
  hourly valid: 39 | daily valid: 22

Common valid hourly variables across all points: 27
['apparent_temperature', 'cape', 'cloud_cover_high', 'cloud_cover_mid', 'dew_point_2m', 'direct_radiation', 'et0_fao_evapotranspiration', 'evapotranspiration', 'global_tilted_irradiance', 'precipitation', 'precipitation_probability', 'shortwave_radiation', 'showers', 'snowfall', 'soil_moisture_0_to_1cm', 'soil_moisture_1_to_3cm', 'soil_moisture_27_to_81cm', 'soil_moisture_9_to_27cm', 'soil_temperature_18cm', 'soil_temperature_6cm', 'surface_pressure', 'weather_code', 'wind_direction_10m', 'wind_direction_80m', 'wind_gusts_10m', 'wind_speed_10m', 'wind_speed_120m']

Common valid daily variables across all points: 14
['apparent_temperature_max', 'a

In [7]:
# Download one payload per location using the common variable schema
payloads_by_location: dict[str, dict] = {}

for location in LOCATIONS:
    loc_id = location["location_id"]

    extra = {}
    if valid_hourly:
        extra["hourly"] = ",".join(valid_hourly)
    if valid_daily:
        extra["daily"] = ",".join(valid_daily)

    if not extra:
        raise ValueError("No valid hourly/daily variables available to download.")

    final_params = location_params(location, **extra)
    payloads_by_location[loc_id] = call_forecast(final_params, timeout=120)
    print(f"Downloaded {loc_id} -> keys: {list(payloads_by_location[loc_id].keys())}")

ConnectionError: ('Connection aborted.', ConnectionResetError(10054, 'Se ha forzado la interrupción de una conexión existente por el host remoto', None, 10054, None))

In [ ]:
# Build geo-tagged DataFrames and concatenate all locations
hourly_frames = []
daily_frames = []

for location in LOCATIONS:
    loc_id = location["location_id"]
    payload = payloads_by_location[loc_id]

    hourly_df_loc = pd.DataFrame(payload.get("hourly", {}))
    daily_df_loc = pd.DataFrame(payload.get("daily", {}))

    if "time" in hourly_df_loc.columns:
        hourly_df_loc = hourly_df_loc.rename(columns={"time": "Timestamp"})
        hourly_df_loc["Timestamp"] = pd.to_datetime(hourly_df_loc["Timestamp"], utc=True, errors="coerce")

    if "time" in daily_df_loc.columns:
        daily_df_loc = daily_df_loc.rename(columns={"time": "date_utc"})
        daily_df_loc["date_utc"] = pd.to_datetime(daily_df_loc["date_utc"], utc=True, errors="coerce")

    hourly_df_loc["location_id"] = loc_id
    hourly_df_loc["latitude"] = location["latitude"]
    hourly_df_loc["longitude"] = location["longitude"]

    daily_df_loc["location_id"] = loc_id
    daily_df_loc["latitude"] = location["latitude"]
    daily_df_loc["longitude"] = location["longitude"]

    hourly_frames.append(hourly_df_loc)
    daily_frames.append(daily_df_loc)

hourly_df = pd.concat(hourly_frames, ignore_index=True) if hourly_frames else pd.DataFrame()
daily_df = pd.concat(daily_frames, ignore_index=True) if daily_frames else pd.DataFrame()

print(f"Hourly combined shape: {hourly_df.shape}")
print(f"Daily combined shape: {daily_df.shape}")
print("Unique locations in hourly:", hourly_df["location_id"].nunique() if not hourly_df.empty else 0)
print("Unique locations in daily:", daily_df["location_id"].nunique() if not daily_df.empty else 0)
display(hourly_df.head(3))
display(daily_df.head(3))

Hourly combined shape: (8760, 45)
Daily combined shape: (365, 24)
Unique locations in hourly: 1
Unique locations in daily: 1


,Timestamp,apparent_temperature,cape,cloud_cover,cloud_cover_high,cloud_cover_low,dew_point_2m,diffuse_radiation,direct_normal_irradiance,direct_radiation,...,wind_direction_120m,wind_direction_180m,wind_direction_80m,wind_gusts_10m,wind_speed_10m,wind_speed_120m,wind_speed_80m,location_id,latitude,longitude
0,2025-01-01 00:00:00+00:00,6.3,None,100,100,98,7.9,0.0,0.0,0.0,...,None,None,None,55.1,25.1,None,None,uk_london,51.5074,-0.1278
1,2025-01-01 01:00:00+00:00,7.0,None,100,100,100,8.4,0.0,0.0,0.0,...,None,None,None,58.3,25.1,None,None,uk_london,51.5074,-0.1278
2,2025-01-01 02:00:00+00:00,7.4,None,100,100,100,8.6,0.0,0.0,0.0,...,None,None,None,60.5,22.9,None,None,uk_london,51.5074,-0.1278


,date_utc,apparent_temperature_max,apparent_temperature_mean,apparent_temperature_min,daylight_duration,et0_fao_evapotranspiration,precipitation_hours,precipitation_probability_max,precipitation_probability_mean,precipitation_probability_min,...,temperature_2m_mean,uv_index_clear_sky_max,uv_index_max,weather_code,wind_direction_10m_dominant,wind_gusts_10m_max,wind_speed_10m_max,location_id,latitude,longitude
0,2025-01-01 00:00:00+00:00,7.5,4.9,1.7,28588.50,0.51,13.0,None,None,None,...,8.8,None,None,61,235,71.6,30.6,uk_london,51.5074,-0.1278
1,2025-01-02 00:00:00+00:00,1.1,-1.4,-4.2,28661.22,0.35,0.0,None,None,None,...,1.9,None,None,3,318,25.6,9.2,uk_london,51.5074,-0.1278
2,2025-01-03 00:00:00+00:00,-0.0,-3.3,-5.2,28739.50,0.33,0.0,None,None,None,...,0.3,None,None,3,253,22.7,10.6,uk_london,51.5074,-0.1278


In [ ]:
hourly_df

,Timestamp,apparent_temperature,cape,cloud_cover,cloud_cover_high,cloud_cover_low,dew_point_2m,diffuse_radiation,direct_normal_irradiance,direct_radiation,...,wind_direction_120m,wind_direction_180m,wind_direction_80m,wind_gusts_10m,wind_speed_10m,wind_speed_120m,wind_speed_80m,location_id,latitude,longitude
0,2025-01-01 00:00:00+00:00,6.3,None,100,100,98,7.9,0.0,0.0,0.0,...,None,None,None,55.1,25.1,None,None,uk_london,51.5074,-0.1278
1,2025-01-01 01:00:00+00:00,7.0,None,100,100,100,8.4,0.0,0.0,0.0,...,None,None,None,58.3,25.1,None,None,uk_london,51.5074,-0.1278
2,2025-01-01 02:00:00+00:00,7.4,None,100,100,100,8.6,0.0,0.0,0.0,...,None,None,None,60.5,22.9,None,None,uk_london,51.5074,-0.1278
3,2025-01-01 03:00:00+00:00,7.5,None,100,100,89,8.4,0.0,0.0,0.0,...,None,None,None,61.9,22.0,None,None,uk_london,51.5074,-0.1278
4,2025-01-01 04:00:00+00:00,7.0,None,100,100,100,8.4,0.0,0.0,0.0,...,None,None,None,58.3,23.4,None,None,uk_london,51.5074,-0.1278
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8755,2025-12-31 19:00:00+00:00,-3.6,None,2,0,2,-1.1,0.0,0.0,0.0,...,None,None,None,22.0,10.7,None,None,uk_london,51.5074,-0.1278
8756,2025-12-31 20:00:00+00:00,-4.2,None,8,0,8,-1.4,0.0,0.0,0.0,...,None,None,None,22.3,10.8,None,None,uk_london,51.5074,-0.1278
8757,2025-12-31 21:00:00+00:00,-4.7,None,5,0,5,-1.6,0.0,0.0,0.0,...,None,None,None,23.4,11.2,None,None,uk_london,51.5074,-0.1278
8758,2025-12-31 22:00:00+00:00,-4.9,None,0,0,0,-1.7,0.0,0.0,0.0,...,None,None,None,23.8,11.4,None,None,uk_london,51.5074,-0.1278


In [ ]:
# Save outputs for geospatial model experimentation
hourly_out = OUTPUT_DIR / "openmeteo_hourly_all_available_multi_pointreal.csv"
daily_out = OUTPUT_DIR / "openmeteo_daily_all_available_multi_pointreal.csv"
valid_hourly_out = OUTPUT_DIR / "openmeteo_valid_hourly_variables_commonreal.csv"
valid_daily_out = OUTPUT_DIR / "openmeteo_valid_daily_variables_commonreal.csv"
valid_by_location_out = OUTPUT_DIR / "openmeteo_valid_variables_by_locationreal.csv"

hourly_df.to_csv(hourly_out, index=False)
daily_df.to_csv(daily_out, index=False)
pd.DataFrame({"hourly_variable": valid_hourly}).to_csv(valid_hourly_out, index=False)
pd.DataFrame({"daily_variable": valid_daily}).to_csv(valid_daily_out, index=False)

rows = []
for location in LOCATIONS:
    loc_id = location["location_id"]
    details = validation_by_location[loc_id]
    rows.append(
        {
            "location_id": loc_id,
            "latitude": location["latitude"],
            "longitude": location["longitude"],
            "valid_hourly_count": len(details["valid_hourly"]),
            "valid_daily_count": len(details["valid_daily"]),
        }
    )
pd.DataFrame(rows).to_csv(valid_by_location_out, index=False)

print(f"Saved: {hourly_out}")
print(f"Saved: {daily_out}")
print(f"Saved: {valid_hourly_out}")
print(f"Saved: {valid_daily_out}")
print(f"Saved: {valid_by_location_out}")

Saved: c:\Users\rkale\OneDrive\Documentos\VStudio\GitHub\Gitlab\GroupProject\25-26_CE903-SP_team03\data\openmeteo_hourly_all_available_multi_point.csv
Saved: c:\Users\rkale\OneDrive\Documentos\VStudio\GitHub\Gitlab\GroupProject\25-26_CE903-SP_team03\data\openmeteo_daily_all_available_multi_point.csv
Saved: c:\Users\rkale\OneDrive\Documentos\VStudio\GitHub\Gitlab\GroupProject\25-26_CE903-SP_team03\data\openmeteo_valid_hourly_variables_common.csv
Saved: c:\Users\rkale\OneDrive\Documentos\VStudio\GitHub\Gitlab\GroupProject\25-26_CE903-SP_team03\data\openmeteo_valid_daily_variables_common.csv
Saved: c:\Users\rkale\OneDrive\Documentos\VStudio\GitHub\Gitlab\GroupProject\25-26_CE903-SP_team03\data\openmeteo_valid_variables_by_location.csv


## Notes
- Variable availability depends on location, model and date range.
- If you want strictly UKMO variables, set `MODEL = "ukmo_uk_deterministic_2km"` and rerun validation.
- For training with 15-minute target data, you can upsample `hourly_df` to 15 minutes and forward-fill/interpolate as done in your TFT notebook.